# Driver Onboarding Pilot — Synthetic Evaluation

**Portfolio demonstration only. All records and outcomes are fictional. No employer source data are used.**

**Business question:** Can a change in driver onboarding improve route performance and short-term driver activity without compromising service quality and safety?

We calculate a simple pre/post and pilot/comparison difference-in-differences (DiD) contrast. This notebook illustrates the method; it does **not** establish causal impact.

## 1. Load data and inspect quality

The synthetic generator creates 24 fictional sites, 4,800 fictional route observations and 1,200 fictional new-driver starts. We check IDs, missingness and the consistency of route late flags.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from analyze_pilot import summaries, did, get_value
routes = pd.read_csv(project_root / 'data' / 'synthetic_routes.csv')
drivers = pd.read_csv(project_root / 'data' / 'synthetic_drivers.csv')
print('Routes:', routes.shape, '| Drivers:', drivers.shape)
print('Missing values:', int(routes.isna().sum().sum() + drivers.isna().sum().sum()))
assert routes.route_id.is_unique and drivers.driver_id.is_unique
assert (routes.late_route_flag == (routes.overrun_minutes > 30).astype(int)).all()
assert set(routes.group.unique()) == {'Pilot', 'Comparison'}
routes.head()

Routes: (4800, 9) | Drivers: (1200, 6)
Missing values: 0


,route_id,site_id,group,period,overrun_minutes,late_route_flag,on_time_dispatch,delivery_completion_pct,safety_proxy_score
0,R000001,S001,Pilot,Pre,18,0,1,99.28,822.1
1,R000002,S001,Pilot,Pre,52,1,1,98.53,794.0
2,R000003,S001,Pilot,Pre,-5,0,1,99.26,817.5
3,R000004,S001,Pilot,Pre,-12,0,1,99.83,783.0
4,R000005,S001,Pilot,Pre,-11,0,1,98.67,826.2


## 2. Define comparable KPIs

- **Late-route rate** = fraction of routes taking >30 minutes beyond plan; lower is better.
- **Day-14 activity** = eligible starters still active 14 days after starting; higher is better.
- **On-time dispatch** and **delivery completion** track operations and service quality.
- **Safety-proxy score** is fictional and *not* an actual safety measurement.

In [2]:
route_stats, driver_stats = summaries(routes, drivers)
print('ROUTE METRICS (all synthetic):')
display(route_stats.round(2))
print('DRIVER METRICS (all synthetic):')
display(driver_stats.round(2))

ROUTE METRICS (all synthetic):


,group,period,n_routes,late_rate,dispatch_rate,completion_pct,safety_score
0,Comparison,Post,1200,20.33,86.42,98.15,795.56
1,Comparison,Pre,1200,19.25,84.67,98.23,798.70
2,Pilot,Post,1200,13.08,91.58,97.41,783.04
3,Pilot,Pre,1200,24.75,87.08,98.17,800.24


DRIVER METRICS (all synthetic):


,group,period,n_drivers,day14_retention
0,Comparison,Post,300,77.67
1,Comparison,Pre,300,77.00
2,Pilot,Post,300,83.33
3,Pilot,Pre,300,71.00


## 3. Calculate the two-period DiD

For any rate metric, in **percentage points (pp)**:

$\mathrm{DiD} = (\mathrm{Pilot}_{post}-\mathrm{Pilot}_{pre}) - (\mathrm{Comparison}_{post}-\mathrm{Comparison}_{pre})$

Negative DiD for late routes is favourable; negative DiD for service quality may be unfavourable. DiD alone does not guarantee causal identification.

In [3]:
metrics = {
    'Late-route rate (pp)': (route_stats, 'late_rate'),
    'Day-14 activity (pp)': (driver_stats, 'day14_retention'),
    'On-time dispatch (pp)': (route_stats, 'dispatch_rate'),
    'Completion (pp)': (route_stats, 'completion_pct'),
    'Safety proxy (points)': (route_stats, 'safety_score'),
}
results = pd.DataFrame([
    {'Metric': name, 'Illustrative DiD': round(did(df, col), 2)}
    for name, (df, col) in metrics.items()
])
display(results)

,Metric,Illustrative DiD
0,Late-route rate (pp),-12.75
1,Day-14 activity (pp),11.67
2,On-time dispatch (pp),2.75
3,Completion (pp),-0.68
4,Safety proxy (points),-14.07


## 4. Communicate changes visually

Use the two lines to distinguish overall time changes from the relative pilot change. These charts show only the **synthetic scenario**.

In [4]:
fig, ax = plt.subplots(figsize=(9, 5))
for group in ['Pilot', 'Comparison']:
    values = [get_value(route_stats, group, p, 'late_rate') for p in ['Pre', 'Post']]
    ax.plot(['Before', 'After'], values, marker='o', linewidth=2.5, label=group)
ax.set_title('Fictional late-route rates | pilot versus comparison')
ax.set_ylabel('Late routes (%)')
ax.grid(axis='y', alpha=.2)
ax.legend()
plt.show()

/tmp/ipykernel_1824/4274783601.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))
for group in ['Pilot', 'Comparison']:
    values = [get_value(driver_stats, group, p, 'day14_retention') for p in ['Pre', 'Post']]
    ax.plot(['Before', 'After'], values, marker='o', linewidth=2.5, label=group)
ax.set_title('Fictional day-14 activity | pilot versus comparison')
ax.set_ylabel('Eligible starters still active (%)')
ax.grid(axis='y', alpha=.2)
ax.legend()
plt.show()

/tmp/ipykernel_1824/4241881022.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Decision-oriented interpretation

In this **synthetic** scenario, the pilot demonstrates improvements in late-route rates and short-term driver activity relative to a comparison group, but service completion and a fictional safety proxy deteriorate. This suggests monitoring before any wider rollout, rather than simply presenting the improvement as a success.

**Limitations to acknowledge:**

- No random station assignment, so selection bias remains possible.
- Only two time periods; **parallel trends cannot be assessed** from this dataset.
- Day-14 activity is not long-term retention or proof of driver competence.
- A safety-proxy score is not an observed safety incident rate.
- No real financial inputs, cost savings or actual business impact are represented.
- Stronger analysis would add multiple pre-periods, clear cohort eligibility, uncertainty intervals, checks for mix changes and site-level clustering.

**Portfolio takeaway:** Transparent assumptions, accurate calculations and discussion of unintended consequences matter just as much as positive headline KPIs.